## HistGradientBoosting Classifier: Job Satisfaction

## Overview
This notebook trains a `HistGradientBoostingClassifier` to predict binary job satisfaction (`satjob`) using the cleaned GSS 2024 dataset combined with O*NET work values and occupation features.

## Data Preparation & Preprocessing
- **Target Binarization:** Converted the 4-point `satjob` scale into a binary outcome where responses 2 through 4 map to `1` and response 1 maps to `0`.
- **Missing Values:** Dropped rows with missing `satjob` values, resulting in a working dataset of **2,782 records** across **28 features** (557 test samples).
- **Categorical Encoding:** Cast the 10 nominal categorical features (`wrkstat`, `wrkslf`, `sex`, `race`, `hispanic`, `born`, `degree`, `marital`, `region`, `union1`) to pandas `category` dtype so `HistGradientBoostingClassifier` could handle them natively (`categorical_features="from_dtype"`) without one-hot expansion.
- **Train/Test Split:** Split the data using a stratified 80/20 train-test ratio (`random_state=42`).

## Model & Hyperparameter Tuning
We trained a `HistGradientBoostingClassifier` and used 5-fold `GridSearchCV` (optimizing for accuracy) to tune tree complexity and regularization:

- **Evaluated Parameters:**
  - `max_iter`: `[100, 200]`
  - `learning_rate`: `[0.02, 0.05, 0.1]`
  - `max_depth`: `[2, 3, 5]`
  - `min_samples_leaf`: `[20, 50, 100]`
  - `l2_regularization`: `[0.0, 1.0, 5.0]`

- **Optimal Configuration Found:**
  - `max_iter`: `200`
  - `learning_rate`: `0.05`
  - `max_depth`: `3`
  - `min_samples_leaf`: `50`
  - `l2_regularization`: `0.0`

## Results & Performance
- **GridSearchCV Best CV Accuracy:** 61.12%
- **Test Set Accuracy:** 58.53%
- **Test Set ROC-AUC:** 0.5789

### Test Set Classification Report Summary
- **Class 0 (`satjob == 1`, n = 233):** Precision: `0.51` | Recall: `0.31` | F1-score: `0.39`
- **Class 1 (`satjob != 1`, n = 324):** Precision: `0.61` | Recall: `0.78` | F1-score: `0.69`
- **Overall:** Weighted F1-score of `0.56` across 557 test samples, with stronger recall on the majority class (`1`) than the minority class (`0`).

In [1]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split

In [3]:
# Load and drop missing targets
df = pd.read_csv("gss2024_onet_cleaned.csv").dropna(subset=["satjob"]).copy()

# Match training_dt target mapping: 1 -> 0, 2-4 -> 1
df["target"] = (df["satjob"] != 1).astype(int)

features = [
    "occ10",
    "indus10",
    "wrkstat",
    "wrkslf",
    "prestg10",
    "sei10",
    "realrinc",
    "rincom16",
    "hrs1",
    "weekswrk",
    "age",
    "sex",
    "race",
    "hispanic",
    "born",
    "educ",
    "degree",
    "marital",
    "childs",
    "region",
    "union1",
    "Achievement",
    "Independence",
    "Recognition",
    "Relationships",
    "Support",
    "Working Conditions",
    "job_zone",
]

X = df[features].copy()
y = df["target"]

categorical_features = [
    "wrkstat",
    "wrkslf",
    "sex",
    "race",
    "hispanic",
    "born",
    "degree",
    "marital",
    "region",
    "union1",
]
for col in categorical_features:
  X[col] = X[col].astype("category")

# Standard 80/20 train/test split (matches training_dt)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
hgb = HistGradientBoostingClassifier(
    categorical_features="from_dtype", random_state=42
)

param_grid = {
    "max_iter": [100, 200],
    "learning_rate": [0.02, 0.05, 0.1],
    "max_depth": [2, 3, 5],
    "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 1.0, 5.0],
}

grid_search = GridSearchCV(
    estimator=hgb,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",  # Matches training_dt's scoring metric
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")
print("Best Params:", grid_search.best_params_)

C:\Users\fummy\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\fummy\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\fummy\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\fummy\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\fummy\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Best CV Accuracy: 0.6112
Best Params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 200, 'min_samples_leaf': 50}


In [7]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test Set ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test Set Accuracy: 0.5853
Test Set ROC-AUC:  0.5789

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.31      0.39       233
           1       0.61      0.78      0.69       324

    accuracy                           0.59       557
   macro avg       0.56      0.55      0.54       557
weighted avg       0.57      0.59      0.56       557

